In [ ]:
import os, sys
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"
sys.argv = [sys.argv[0]]


In [ ]:
import argparse
import json
from pathlib import Path

import cv2
import h5py
import joblib
import numpy as np
import pandas as pd
import polars as pl

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor

from catboost import CatBoostClassifier
from tqdm import tqdm


# ============================================================
# Arguments
# ============================================================

parser = argparse.ArgumentParser()

parser.add_argument(
    "--batch-size",
    type=int,
    default=32,
)

parser.add_argument(
    "--num-workers",
    type=int,
    default=0,
)

parser.add_argument(
    "--amp",
    action="store_true",
    help="Use FP16 automatic mixed precision for CNN inference",
)

parser.add_argument(
    "--output",
    type=str,
    default="/kaggle/working/submission.csv",
)

parser.set_defaults(amp=True)

args = parser.parse_args()


# ============================================================
# Paths
# ============================================================

ROOT = Path.cwd()

# ============================================================
# Kaggle input path discovery
# Competition sources are currently mounted under:
# /kaggle/input/competitions/<slug>/
# ============================================================

INPUT_ROOT = Path("/kaggle/input")

competition_candidates = [
    INPUT_ROOT / "competitions" / "isic-2024-challenge",
    INPUT_ROOT / "isic-2024-challenge",
]

DATA_DIR = None

for candidate in competition_candidates:
    if (
        (candidate / "test-metadata.csv").exists()
        and (candidate / "test-image.hdf5").exists()
    ):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    print("Available /kaggle/input structure:")
    for x in sorted(INPUT_ROOT.iterdir()):
        print(" ", x)

    raise FileNotFoundError(
        "Cannot locate ISIC 2024 competition data."
    )

print("Competition data:", DATA_DIR)

H5_PATH = DATA_DIR / "test-image.hdf5"
META_PATH = DATA_DIR / "test-metadata.csv"
SAMPLE_PATH = DATA_DIR / "sample_submission.csv"

# ============================================================
# Baseline 4A assets
# ============================================================

baseline_candidates = [
    INPUT_ROOT / "isic2024-winner-original-baseline4a",
    INPUT_ROOT / "datasets" / "lsn0000" / "isic2024-winner-original-baseline4a",
]

BASELINE_DIR = None

for candidate in baseline_candidates:
    if (
        (candidate / "models").exists()
        and (candidate / "preprocessing").exists()
        and (candidate / "metadata").exists()
    ):
        BASELINE_DIR = candidate
        break

if BASELINE_DIR is None:
    raise FileNotFoundError(
        "Cannot locate Baseline 4A assets."
    )


# ============================================================
# Team model configuration
# ============================================================
#
# BASELINE_DIR:
#   Shared CNN weights, preprocessing, and metadata.
#
# USER_MODEL_DIR:
#   GBT models used for the current experiment.
#
# To reproduce the provided baseline, leave this unchanged.
#
# For your own experiment:
#   1. Train your own GBT models.
#   2. Upload them as a Kaggle Dataset.
#   3. Add that Dataset as notebook input.
#   4. Change USER_MODEL_DIR to your Dataset path.
#
# Example:
#
# USER_MODEL_DIR = Path(
#     "/kaggle/input/datasets/<username>/<dataset-name>"
# )
#
USER_MODEL_DIR = BASELINE_DIR


# Original winner CNN weights are packaged locally
MODEL_DIR = BASELINE_DIR / "models"

CATBOOST_DIR = (
    USER_MODEL_DIR /
    "catboost_4d"
)

PREP_DIR = (
    BASELINE_DIR /
    "preprocessing"
)

META_DIR = (
    BASELINE_DIR /
    "metadata"
)


# Fold-0 OOF scaling used for winner test inference
with open(
    META_DIR /
    "cnn_scaling_stats.json",
    "r",
    encoding="utf-8",
) as f:
    cnn_scaling = json.load(f)


EVA_FOLD0_MEAN = float(
    cnn_scaling["eva"]["mean"]
)

EVA_FOLD0_STD = float(
    cnn_scaling["eva"]["std"]
)

EDGE_FOLD0_MEAN = float(
    cnn_scaling["edgenext"]["mean"]
)

EDGE_FOLD0_STD = float(
    cnn_scaling["edgenext"]["std"]
)


print(
    "Baseline 4A assets:",
    BASELINE_DIR
)

print(
    "Local model assets:",
    MODEL_DIR
)

print(
    "EVA fold0 mean/std:",
    EVA_FOLD0_MEAN,
    EVA_FOLD0_STD,
)

print(
    "Edge fold0 mean/std:",
    EDGE_FOLD0_MEAN,
    EDGE_FOLD0_STD,
)


# ============================================================
# Device
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("=" * 90)
print("BASELINE 4D - NOISE + RANDOM STRENGTH INFERENCE")
print("=" * 90)

print("Device:", device)


# ============================================================
# Model definitions
#
# IMPORTANT:
# pretrained=False prevents Internet access.
# ============================================================

EVA_MODEL_NAME = (
    "eva02_small_patch14_336."
    "mim_in22k_ft_in1k"
)

EDGENEXT_MODEL_NAME = (
    "edgenext_base.in21k_ft_in1k"
)


class ISICModel(nn.Module):

    def __init__(
        self,
        model_name,
        num_classes=1,
    ):
        super().__init__()

        self.model = timm.create_model(
            model_name,
            pretrained=False,
        )

        in_features = (
            self.model.head.in_features
        )

        self.model.head = nn.Linear(
            in_features,
            num_classes,
        )

        if num_classes == 1:
            self.activation = nn.Sigmoid()
        else:
            self.activation = nn.Softmax(
                dim=1
            )

    def forward(self, x):
        return self.activation(
            self.model(x)
        )


class ISICModelEdgenext(nn.Module):

    def __init__(
        self,
        model_name,
        num_classes=1,
    ):
        super().__init__()

        self.model = timm.create_model(
            model_name,
            pretrained=False,
            num_classes=num_classes,
            global_pool="avg",
        )

        self.activation = nn.Sigmoid()

    def forward(self, x):
        return self.activation(
            self.model(x)
        )


# ============================================================
# Load state dict
# ============================================================

def load_state_model(
    model,
    path,
):

    state = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    if isinstance(
        state,
        nn.Module,
    ):
        state = state.state_dict()

    if isinstance(state, dict):

        if (
            "state_dict" in state
            and isinstance(
                state["state_dict"],
                dict,
            )
        ):
            state = state[
                "state_dict"
            ]

        elif (
            "model_state_dict" in state
            and isinstance(
                state["model_state_dict"],
                dict,
            )
        ):
            state = state[
                "model_state_dict"
            ]

    if (
        isinstance(state, dict)
        and any(
            str(k).startswith(
                "module."
            )
            for k in state.keys()
        )
    ):

        state = {
            str(k).removeprefix(
                "module."
            ): v
            for k, v
            in state.items()
        }

    model.load_state_dict(
        state,
        strict=True,
    )

    model = model.to(device)
    model.eval()

    print(
        "Loaded:",
        Path(path).name
    )

    return model


# ============================================================
# Validation transforms
# Exact mean/std from original augmentations.py
# ============================================================

MEAN = [
    0.4815,
    0.4578,
    0.4082,
]

STD = [
    0.2686,
    0.2613,
    0.2758,
]


transform_336 = A.Compose([
    A.Resize(
        336,
        336,
    ),
    A.Normalize(
        mean=MEAN,
        std=STD,
    ),
    ToTensorV2(),
])


transform_256 = A.Compose([
    A.Resize(
        256,
        256,
    ),
    A.Normalize(
        mean=MEAN,
        std=STD,
    ),
    ToTensorV2(),
])


# ============================================================
# HDF5 dataset
# Decode JPEG bytes:
# cv2.imdecode -> BGR -> RGB
# same convention as training Dataset.
# ============================================================

class TestH5Dataset(Dataset):

    def __init__(
        self,
        h5_path,
        isic_ids,
    ):

        self.h5_path = str(h5_path)

        self.isic_ids = list(
            isic_ids
        )

        self.h5 = None

    def __len__(self):
        return len(self.isic_ids)

    def _get_h5(self):

        if self.h5 is None:
            self.h5 = h5py.File(
                self.h5_path,
                "r",
            )

        return self.h5

    def __getitem__(self, idx):

        isic_id = self.isic_ids[idx]

        h5 = self._get_h5()

        # ----------------------------------------------------
        # Hidden-test robustness:
        # one missing/corrupted image must not terminate the
        # entire Kaggle submission.
        # ----------------------------------------------------

        try:

            if isic_id not in h5:
                raise KeyError(
                    f"{isic_id} not found in HDF5"
                )

            jpeg_bytes = h5[
                isic_id
            ][()]

            if isinstance(
                jpeg_bytes,
                np.ndarray,
            ):
                encoded = (
                    jpeg_bytes
                    .astype(
                        np.uint8,
                        copy=False,
                    )
                    .reshape(-1)
                )
            else:
                encoded = np.frombuffer(
                    jpeg_bytes,
                    dtype=np.uint8,
                )

            if encoded.size == 0:
                raise RuntimeError(
                    "Empty JPEG byte array"
                )

            image = cv2.imdecode(
                encoded,
                cv2.IMREAD_COLOR,
            )

            if image is None:
                raise RuntimeError(
                    "cv2.imdecode returned None"
                )

            if (
                image.ndim != 3
                or image.shape[2] != 3
            ):
                raise RuntimeError(
                    f"Unexpected image shape "
                    f"{image.shape}"
                )

            image = cv2.cvtColor(
                image,
                cv2.COLOR_BGR2RGB,
            )

        except Exception as e:

            print(
                f"WARNING: image fallback for "
                f"{isic_id}: "
                f"{type(e).__name__}: {e}",
                flush=True,
            )

            # Neutral placeholder image.
            # Used only if one hidden image cannot be read.
            image = np.full(
                (336, 336, 3),
                127,
                dtype=np.uint8,
            )


        img336 = transform_336(
            image=image
        )["image"]

        img256 = transform_256(
            image=image
        )["image"]

        return {
            "isic_id": isic_id,
            "image336": img336,
            "image256": img256,
        }


# ============================================================
# CNN inference
# ============================================================

def run_cnn_inference(
    test_meta,
):

    print("\n" + "=" * 90)
    print("LOADING IMAGE MODELS")
    print("=" * 90)

    # External EVA: 3 classes
    external_eva = load_state_model(
        ISICModel(
            EVA_MODEL_NAME,
            num_classes=3,
        ),
        MODEL_DIR /
        "external_eva" /
        "external_eva_best.pth",
    )


    # Competition EVA × 5
    eva_models = []

    for fold in range(5):

        model = load_state_model(
            ISICModel(
                EVA_MODEL_NAME,
                num_classes=1,
            ),
            MODEL_DIR /
            "competition_eva" /
            f"model_fold{fold}.pth",
        )

        eva_models.append(model)


    # EdgeNeXt × 5
    edge_models = []

    for fold in range(5):

        model = load_state_model(
            ISICModelEdgenext(
                EDGENEXT_MODEL_NAME,
                num_classes=1,
            ),
            MODEL_DIR /
            "edgenext" /
            f"model_fold{fold}.pth",
        )

        edge_models.append(model)


    dataset = TestH5Dataset(
        H5_PATH,
        test_meta["isic_id"].tolist(),
    )

    loader = DataLoader(
        dataset,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=True,
        persistent_workers=(
            args.num_workers > 0
        ),
    )


    rows = []

    print("\n" + "=" * 90)
    print("CNN INFERENCE")
    print("=" * 90)

    with torch.inference_mode():

        for batch in tqdm(loader):

            ids = batch["isic_id"]

            x336 = (
                batch["image336"]
                .to(
                    device,
                    non_blocking=True,
                )
            )

            x256 = (
                batch["image256"]
                .to(
                    device,
                    non_blocking=True,
                )
            )


            # --------------------------------------------
            # CNN forward
            # Optional FP16 AMP for faster inference
            # --------------------------------------------

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(
                    args.amp
                    and device.type == "cuda"
                ),
            ):

                # External EVA
                # class 0 = nevus
                # class 1 = bkl
                # class 2 = melanoma
                old_pred = external_eva(x336)

                # Competition EVA x5
                # Each fold prediction is standardized using
                # FOLD-0 OOF mean/std before averaging.
                eva_pred = 0.0

                for model in eva_models:

                    p = model(x336)

                    p = (
                        p - EVA_FOLD0_MEAN
                    ) / EVA_FOLD0_STD

                    eva_pred = (
                        eva_pred + p
                    )

                eva_pred = (
                    eva_pred /
                    len(eva_models)
                )


                # EdgeNeXt x5
                edge_pred = 0.0

                for model in edge_models:

                    p = model(x256)

                    p = (
                        p - EDGE_FOLD0_MEAN
                    ) / EDGE_FOLD0_STD

                    edge_pred = (
                        edge_pred + p
                    )

                edge_pred = (
                    edge_pred /
                    len(edge_models)
                )


            old_pred = (
                old_pred
                .detach()
                .float()
                .cpu()
                .numpy()
            )

            eva_pred = (
                eva_pred
                .squeeze(1)
                .detach()
                .float()
                .cpu()
                .numpy()
            )

            edge_pred = (
                edge_pred
                .squeeze(1)
                .detach()
                .float()
                .cpu()
                .numpy()
            )


            for i, isic_id in enumerate(ids):

                rows.append({
                    "isic_id":
                        isic_id,

                    "old_set_0":
                        float(
                            old_pred[i, 0]
                        ),

                    "old_set_1":
                        float(
                            old_pred[i, 1]
                        ),

                    "old_set_2":
                        float(
                            old_pred[i, 2]
                        ),

                    "oof_eva_score":
                        float(
                            eva_pred[i]
                        ),

                    "oof_edgenext_score":
                        float(
                            edge_pred[i]
                        ),
                })


    pred_df = pd.DataFrame(rows)


    print("\nCNN predictions:")
    print("Rows:", len(pred_df))
    print(
        pred_df.head(3).to_string(
            index=False
        )
    )

    return pred_df


# ============================================================
# Original top-model metadata features
# ============================================================

err = 1e-5


num_cols = [
    "age_approx",
    "clin_size_long_diam_mm",
    "tbp_lv_A",
    "tbp_lv_Aext",
    "tbp_lv_B",
    "tbp_lv_Bext",
    "tbp_lv_C",
    "tbp_lv_Cext",
    "tbp_lv_H",
    "tbp_lv_Hext",
    "tbp_lv_L",
    "tbp_lv_Lext",
    "tbp_lv_areaMM2",
    "tbp_lv_area_perim_ratio",
    "tbp_lv_color_std_mean",
    "tbp_lv_deltaA",
    "tbp_lv_deltaB",
    "tbp_lv_deltaL",
    "tbp_lv_deltaLB",
    "tbp_lv_deltaLBnorm",
    "tbp_lv_eccentricity",
    "tbp_lv_minorAxisMM",
    "tbp_lv_nevi_confidence",
    "tbp_lv_norm_border",
    "tbp_lv_norm_color",
    "tbp_lv_perimeterMM",
    "tbp_lv_radial_color_std_max",
    "tbp_lv_stdL",
    "tbp_lv_stdLExt",
    "tbp_lv_symm_2axis",
    "tbp_lv_symm_2axis_angle",
    "tbp_lv_x",
    "tbp_lv_y",
    "tbp_lv_z",
]


new_num_cols = [
    "lesion_size_ratio",
    "lesion_shape_index",
    "hue_contrast",
    "luminance_contrast",
    "lesion_color_difference",
    "border_complexity",
    "color_uniformity",
    "position_distance_3d",
    "perimeter_to_area_ratio",
    "area_to_perimeter_ratio",
    "lesion_visibility_score",
    "symmetry_border_consistency",
    "consistency_symmetry_border",
    "color_consistency",
    "consistency_color",
    "size_age_interaction",
    "hue_color_std_interaction",
    "lesion_severity_index",
    "shape_complexity_index",
    "color_contrast_index",
    "log_lesion_area",
    "normalized_lesion_size",
    "mean_hue_difference",
    "std_dev_contrast",
    "color_shape_composite_index",
    "lesion_orientation_3d",
    "overall_color_difference",
    "symmetry_perimeter_interaction",
    "comprehensive_lesion_index",
    "color_variance_ratio",
    "border_color_interaction",
    "border_color_interaction_2",
    "size_color_contrast_ratio",
    "age_normalized_nevi_confidence",
    "age_normalized_nevi_confidence_2",
    "color_asymmetry_index",
    "volume_approximation_3d",
    "color_range",
    "shape_color_consistency",
    "border_length_ratio",
    "age_size_symmetry_index",
    "index_age_size_symmetry",
]


raw_cat_cols = [
    "sex",
    "anatom_site_general",
    "tbp_tile_type",
    "tbp_lv_location",
    "tbp_lv_location_simple",
    "attribution",
]


def read_data(path):

    return (
        pl.read_csv(path)

        .with_columns(
            pl.col("age_approx")
            .cast(pl.String)
            .replace("NA", np.nan)
            .cast(pl.Float64),
        )

        .with_columns(
            pl.col(pl.Float64)
            .fill_nan(
                pl.col(
                    pl.Float64
                ).median()
            ),
        )

        # Hidden-test robustness:
        # CSV blanks may become Polars null rather than NaN.
        .with_columns(
            pl.col(pl.Float64)
            .fill_null(
                pl.col(
                    pl.Float64
                ).median()
            ),
        )

        .with_columns(
            lesion_size_ratio=
                pl.col(
                    "tbp_lv_minorAxisMM"
                )
                /
                pl.col(
                    "clin_size_long_diam_mm"
                ),

            lesion_shape_index=
                pl.col(
                    "tbp_lv_areaMM2"
                )
                /
                (
                    pl.col(
                        "tbp_lv_perimeterMM"
                    ) ** 2
                ),

            hue_contrast=
                (
                    pl.col("tbp_lv_H")
                    -
                    pl.col("tbp_lv_Hext")
                ).abs(),

            luminance_contrast=
                (
                    pl.col("tbp_lv_L")
                    -
                    pl.col("tbp_lv_Lext")
                ).abs(),

            lesion_color_difference=
                (
                    pl.col(
                        "tbp_lv_deltaA"
                    ) ** 2
                    +
                    pl.col(
                        "tbp_lv_deltaB"
                    ) ** 2
                    +
                    pl.col(
                        "tbp_lv_deltaL"
                    ) ** 2
                ).sqrt(),

            border_complexity=
                pl.col(
                    "tbp_lv_norm_border"
                )
                +
                pl.col(
                    "tbp_lv_symm_2axis"
                ),

            color_uniformity=
                pl.col(
                    "tbp_lv_color_std_mean"
                )
                /
                (
                    pl.col(
                        "tbp_lv_radial_color_std_max"
                    )
                    + err
                ),
        )

        .with_columns(
            position_distance_3d=
                (
                    pl.col(
                        "tbp_lv_x"
                    ) ** 2
                    +
                    pl.col(
                        "tbp_lv_y"
                    ) ** 2
                    +
                    pl.col(
                        "tbp_lv_z"
                    ) ** 2
                ).sqrt(),

            perimeter_to_area_ratio=
                pl.col(
                    "tbp_lv_perimeterMM"
                )
                /
                pl.col(
                    "tbp_lv_areaMM2"
                ),

            area_to_perimeter_ratio=
                pl.col(
                    "tbp_lv_areaMM2"
                )
                /
                pl.col(
                    "tbp_lv_perimeterMM"
                ),

            lesion_visibility_score=
                pl.col(
                    "tbp_lv_deltaLBnorm"
                )
                +
                pl.col(
                    "tbp_lv_norm_color"
                ),

            combined_anatomical_site=
                pl.col(
                    "anatom_site_general"
                )
                +
                "_"
                +
                pl.col(
                    "tbp_lv_location"
                ),

            symmetry_border_consistency=
                pl.col(
                    "tbp_lv_symm_2axis"
                )
                *
                pl.col(
                    "tbp_lv_norm_border"
                ),

            consistency_symmetry_border=
                (
                    pl.col(
                        "tbp_lv_symm_2axis"
                    )
                    *
                    pl.col(
                        "tbp_lv_norm_border"
                    )
                )
                /
                (
                    pl.col(
                        "tbp_lv_symm_2axis"
                    )
                    +
                    pl.col(
                        "tbp_lv_norm_border"
                    )
                ),
        )

        .with_columns(
            color_consistency=
                pl.col("tbp_lv_stdL")
                /
                pl.col("tbp_lv_Lext"),

            consistency_color=
                (
                    pl.col("tbp_lv_stdL")
                    *
                    pl.col("tbp_lv_Lext")
                )
                /
                (
                    pl.col("tbp_lv_stdL")
                    +
                    pl.col("tbp_lv_Lext")
                ),

            size_age_interaction=
                pl.col(
                    "clin_size_long_diam_mm"
                )
                *
                pl.col(
                    "age_approx"
                ),

            hue_color_std_interaction=
                pl.col("tbp_lv_H")
                *
                pl.col(
                    "tbp_lv_color_std_mean"
                ),

            lesion_severity_index=
                (
                    pl.col(
                        "tbp_lv_norm_border"
                    )
                    +
                    pl.col(
                        "tbp_lv_norm_color"
                    )
                    +
                    pl.col(
                        "tbp_lv_eccentricity"
                    )
                ) / 3,

            shape_complexity_index=
                pl.col(
                    "border_complexity"
                )
                +
                pl.col(
                    "lesion_shape_index"
                ),

            color_contrast_index=
                pl.col(
                    "tbp_lv_deltaA"
                )
                +
                pl.col(
                    "tbp_lv_deltaB"
                )
                +
                pl.col(
                    "tbp_lv_deltaL"
                )
                +
                pl.col(
                    "tbp_lv_deltaLBnorm"
                ),
        )

        .with_columns(
            log_lesion_area=
                (
                    pl.col(
                        "tbp_lv_areaMM2"
                    )
                    + 1
                ).log(),

            normalized_lesion_size=
                pl.col(
                    "clin_size_long_diam_mm"
                )
                /
                pl.col(
                    "age_approx"
                ),

            mean_hue_difference=
                (
                    pl.col(
                        "tbp_lv_H"
                    )
                    +
                    pl.col(
                        "tbp_lv_Hext"
                    )
                ) / 2,

            std_dev_contrast=
                (
                    (
                        pl.col(
                            "tbp_lv_deltaA"
                        ) ** 2
                        +
                        pl.col(
                            "tbp_lv_deltaB"
                        ) ** 2
                        +
                        pl.col(
                            "tbp_lv_deltaL"
                        ) ** 2
                    )
                    / 3
                ).sqrt(),

            color_shape_composite_index=
                (
                    pl.col(
                        "tbp_lv_color_std_mean"
                    )
                    +
                    pl.col(
                        "tbp_lv_area_perim_ratio"
                    )
                    +
                    pl.col(
                        "tbp_lv_symm_2axis"
                    )
                ) / 3,

            lesion_orientation_3d=
                pl.arctan2(
                    pl.col(
                        "tbp_lv_y"
                    ),
                    pl.col(
                        "tbp_lv_x"
                    ),
                ),

            overall_color_difference=
                (
                    pl.col(
                        "tbp_lv_deltaA"
                    )
                    +
                    pl.col(
                        "tbp_lv_deltaB"
                    )
                    +
                    pl.col(
                        "tbp_lv_deltaL"
                    )
                ) / 3,
        )

        .with_columns(
            symmetry_perimeter_interaction=
                pl.col(
                    "tbp_lv_symm_2axis"
                )
                *
                pl.col(
                    "tbp_lv_perimeterMM"
                ),

            comprehensive_lesion_index=
                (
                    pl.col(
                        "tbp_lv_area_perim_ratio"
                    )
                    +
                    pl.col(
                        "tbp_lv_eccentricity"
                    )
                    +
                    pl.col(
                        "tbp_lv_norm_color"
                    )
                    +
                    pl.col(
                        "tbp_lv_symm_2axis"
                    )
                ) / 4,

            color_variance_ratio=
                pl.col(
                    "tbp_lv_color_std_mean"
                )
                /
                pl.col(
                    "tbp_lv_stdLExt"
                ),

            border_color_interaction=
                pl.col(
                    "tbp_lv_norm_border"
                )
                *
                pl.col(
                    "tbp_lv_norm_color"
                ),

            border_color_interaction_2=
                (
                    pl.col(
                        "tbp_lv_norm_border"
                    )
                    *
                    pl.col(
                        "tbp_lv_norm_color"
                    )
                )
                /
                (
                    pl.col(
                        "tbp_lv_norm_border"
                    )
                    +
                    pl.col(
                        "tbp_lv_norm_color"
                    )
                ),

            size_color_contrast_ratio=
                pl.col(
                    "clin_size_long_diam_mm"
                )
                /
                pl.col(
                    "tbp_lv_deltaLBnorm"
                ),

            age_normalized_nevi_confidence=
                pl.col(
                    "tbp_lv_nevi_confidence"
                )
                /
                pl.col(
                    "age_approx"
                ),

            age_normalized_nevi_confidence_2=
                (
                    pl.col(
                        "clin_size_long_diam_mm"
                    ) ** 2
                    +
                    pl.col(
                        "age_approx"
                    ) ** 2
                ).sqrt(),

            color_asymmetry_index=
                pl.col(
                    "tbp_lv_radial_color_std_max"
                )
                *
                pl.col(
                    "tbp_lv_symm_2axis"
                ),
        )

        .with_columns(
            volume_approximation_3d=
                pl.col(
                    "tbp_lv_areaMM2"
                )
                *
                (
                    pl.col(
                        "tbp_lv_x"
                    ) ** 2
                    +
                    pl.col(
                        "tbp_lv_y"
                    ) ** 2
                    +
                    pl.col(
                        "tbp_lv_z"
                    ) ** 2
                ).sqrt(),

            color_range=
                (
                    pl.col(
                        "tbp_lv_L"
                    )
                    -
                    pl.col(
                        "tbp_lv_Lext"
                    )
                ).abs()
                +
                (
                    pl.col(
                        "tbp_lv_A"
                    )
                    -
                    pl.col(
                        "tbp_lv_Aext"
                    )
                ).abs()
                +
                (
                    pl.col(
                        "tbp_lv_B"
                    )
                    -
                    pl.col(
                        "tbp_lv_Bext"
                    )
                ).abs(),

            shape_color_consistency=
                pl.col(
                    "tbp_lv_eccentricity"
                )
                *
                pl.col(
                    "tbp_lv_color_std_mean"
                ),

            border_length_ratio=
                pl.col(
                    "tbp_lv_perimeterMM"
                )
                /
                (
                    2
                    * np.pi
                    *
                    (
                        pl.col(
                            "tbp_lv_areaMM2"
                        )
                        / np.pi
                    ).sqrt()
                ),

            age_size_symmetry_index=
                pl.col(
                    "age_approx"
                )
                *
                pl.col(
                    "clin_size_long_diam_mm"
                )
                *
                pl.col(
                    "tbp_lv_symm_2axis"
                ),

            index_age_size_symmetry=
                pl.col(
                    "age_approx"
                )
                *
                pl.col(
                    "tbp_lv_areaMM2"
                )
                *
                pl.col(
                    "tbp_lv_symm_2axis"
                ),
        )

        # ----------------------------------------------------
        # Hidden-test robustness:
        # engineered ratios can produce +/-inf when a
        # denominator is zero. Convert them to missing values
        # and impute before patient-level normalization.
        # ----------------------------------------------------

        .with_columns(
            pl.col(pl.Float64)
            .replace(
                [np.inf, -np.inf],
                None,
            ),
        )

        .with_columns(
            pl.col(pl.Float64)
            .fill_nan(
                pl.col(
                    pl.Float64
                ).median()
            ),
        )

        .with_columns(
            pl.col(pl.Float64)
            .fill_null(
                pl.col(
                    pl.Float64
                ).median()
            ),
        )

        .with_columns(
            (
                (
                    pl.col(col)
                    -
                    pl.col(col)
                    .mean()
                    .over("patient_id")
                )
                /
                (
                    pl.col(col)
                    .std()
                    .over("patient_id")
                    + err
                )
            ).alias(
                f"{col}_patient_norm"
            )

            for col in (
                num_cols
                +
                new_num_cols
            )
        )

        .with_columns(
            count_per_patient=
                pl.col(
                    "isic_id"
                )
                .count()
                .over(
                    "patient_id"
                ),
        )

        .with_columns(
            tbp_lv_areaMM2_patient=
                pl.col(
                    "tbp_lv_areaMM2"
                )
                .sum()
                .over(
                    "patient_id"
                ),
        )

        .with_columns(
            tbp_lv_areaMM2_bp=
                pl.col(
                    "tbp_lv_areaMM2"
                )
                .sum()
                .over([
                    "patient_id",
                    "anatom_site_general",
                ]),
        )

        .with_columns(
            pl.col(
                raw_cat_cols
            ).cast(
                pl.Categorical
            ),
        )

        .to_pandas()
        .set_index(
            "isic_id"
        )
    )


# ============================================================
# LOF
# ============================================================

def get_lof_score(
    df,
    top_lof_features,
):

    # ----------------------------------------------------
    # Hidden-test robustness:
    # LOF / sklearn must not receive inf or unresolved NaN.
    # Normal finite values are unchanged.
    # ----------------------------------------------------

    lof_input = (
        df[
            top_lof_features
        ]
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .copy()
    )

    lof_input = (
        lof_input
        .fillna(
            lof_input.median(
                numeric_only=True
            )
        )
        .fillna(0.0)
    )

    scaled_array = (
        StandardScaler()
        .fit_transform(
            lof_input
        )
    )

    outlier_factors = []

    for patient in tqdm(
        df["patient_id"].unique(),
        desc="LOF",
    ):

        mask = (
            df["patient_id"]
            == patient
        ).values

        n = int(mask.sum())

        if n < 3:
            continue

        patient_emb = (
            scaled_array[mask]
        )

        clf = LocalOutlierFactor(
            n_neighbors=min(
                n,
                30,
            )
        )

        clf.fit_predict(
            patient_emb
        )

        outlier_factors.append(
            pd.DataFrame({
                "isic_id":
                    df[
                        mask
                    ].index.values,

                "of":
                    clf
                    .negative_outlier_factor_,
            })
        )


    if outlier_factors:

        outlier_factors = (
            pd.concat(
                outlier_factors
            )
            .reset_index(
                drop=True
            )
        )

        df = (
            df.reset_index()
            .merge(
                outlier_factors,
                how="left",
                on="isic_id",
            )
            .set_index(
                "isic_id"
            )
        )

    else:
        df["of"] = np.nan


    df["of"] = (
        df["of"]
        .fillna(-1)
    )

    return df


# ============================================================
# Build CatBoost test features
#
# Public repository does NOT contain original test-time code.
#
# Reconstruction choices:
# - patient-relative features: computed within hidden test
# - attribution/location/age CNN z-scores: computed within test
# - KMeans model + cluster mean/std: frozen from training
# ============================================================

def build_test_features(
    cnn_pred,
):

    print("\n" + "=" * 90)
    print("TEST FEATURE ENGINEERING")
    print("=" * 90)


    with open(
        PREP_DIR /
        "inference_config.json",
        "r",
        encoding="utf-8",
    ) as f:
        config = json.load(f)


    top_lof_features = (
        config[
            "top_lof_features"
        ]
    )

    cluster_cols = (
        config[
            "cluster_cols"
        ]
    )

    model_features = (
        config[
            "model_features"
        ]
    )

    onehot_cols = (
        config[
            "onehot_cols"
        ]
    )


    # --------------------------------------------
    # Original metadata feature engineering
    # --------------------------------------------

    df = read_data(
        META_PATH
    )


    # --------------------------------------------
    # Original OneHotEncoder fitted on TRAIN
    # --------------------------------------------

    # Portable one-hot encoding.
    # Avoid sklearn-version-dependent pickle loading.
    with open(
        PREP_DIR / "onehot_categories.json",
        "r",
        encoding="utf-8",
    ) as f:
        saved_categories = json.load(f)

    onehot_parts = []

    for col, categories in zip(
        raw_cat_cols,
        saved_categories,
    ):
        values = df[col].astype(object).to_numpy()

        block = np.zeros(
            (len(df), len(categories)),
            dtype=np.int32,
        )

        for j, category in enumerate(categories):
            if category is None:
                block[:, j] = pd.isna(values).astype(np.int32)
            else:
                block[:, j] = (values == category).astype(np.int32)

        onehot_parts.append(block)

    onehot = np.concatenate(
        onehot_parts,
        axis=1,
    )

    if onehot.shape[1] != len(onehot_cols):
        raise RuntimeError(
            f"Portable one-hot mismatch: "
            f"{onehot.shape[1]} vs {len(onehot_cols)}"
        )


    onehot_df = pd.DataFrame(
        onehot,
        index=df.index,
        columns=onehot_cols,
    )


    for c in onehot_cols:
        df[c] = (
            onehot_df[c]
            .astype("category")
        )


    # --------------------------------------------
    # LOF / Ugly Duckling
    # --------------------------------------------

    df = get_lof_score(
        df,
        top_lof_features,
    )


    # --------------------------------------------
    # Merge CNN predictions
    # --------------------------------------------

    df = (
        df.reset_index()
        .merge(
            cnn_pred,
            how="left",
            on="isic_id",
            validate="one_to_one",
        )
    )


    cnn_cols = [
        "old_set_0",
        "old_set_1",
        "old_set_2",
        "oof_eva_score",
        "oof_edgenext_score",
    ]

    for c in cnn_cols:

        if df[c].isna().any():
            raise RuntimeError(
                f"Missing CNN score: {c}"
            )


    # --------------------------------------------
    # External EVA patient-relative scores
    # --------------------------------------------

    patient_old = (
        df.groupby(
            "patient_id"
        )
        .agg(
            old_set_0_patient_mean=(
                "old_set_0",
                "mean",
            ),
            old_set_1_patient_mean=(
                "old_set_1",
                "mean",
            ),
            old_set_2_patient_mean=(
                "old_set_2",
                "mean",
            ),
        )
        .reset_index()
    )


    df = df.merge(
        patient_old,
        how="left",
        on="patient_id",
    )


    df["old_set_0_m"] = (
        df["old_set_0"]
        /
        df[
            "old_set_0_patient_mean"
        ]
    )

    df["old_set_1_m"] = (
        df["old_set_1"]
        /
        df[
            "old_set_1_patient_mean"
        ]
    )

    df["old_set_2_m"] = (
        df["old_set_2"]
        /
        df[
            "old_set_2_patient_mean"
        ]
    )


    # --------------------------------------------
    # EVA patient relative
    # --------------------------------------------

    eva_mean = (
        df.groupby(
            "patient_id"
        )[
            "oof_eva_score"
        ]
        .transform(
            "mean"
        )
    )

    df["oof_eva_score_m"] = (
        df[
            "oof_eva_score"
        ]
        /
        eva_mean
    )


    # --------------------------------------------
    # EdgeNeXt patient relative
    # --------------------------------------------

    edge_mean = (
        df.groupby(
            "patient_id"
        )[
            "oof_edgenext_score"
        ]
        .transform(
            "mean"
        )
    )

    df[
        "oof_edgenext_score_m"
    ] = (
        df[
            "oof_edgenext_score"
        ]
        /
        edge_mean
    )


    # --------------------------------------------
    # Original train code grouped by:
    # attribution + location + age
    #
    # We mirror that operation on test.
    # --------------------------------------------

    group_keys = [
        "attribution",
        "tbp_lv_location",
        "age_approx",
    ]


    grp = (
        df.groupby(
            group_keys
        )
        .agg(
            oof_edgenext_score_mean_tmp=(
                "oof_edgenext_score",
                "mean",
            ),

            oof_edgenext_score_std_tmp=(
                "oof_edgenext_score",
                "std",
            ),

            oof_eva_score_mean_tmp=(
                "oof_eva_score",
                "mean",
            ),

            oof_eva_score_std_tmp=(
                "oof_eva_score",
                "std",
            ),
        )
        .reset_index()
    )


    df = df.merge(
        grp,
        how="left",
        on=group_keys,
    )


    df[
        "oof_edgenext_score_mean_tmp"
    ] = (
        (
            df[
                "oof_edgenext_score"
            ]
            -
            df[
                "oof_edgenext_score_mean_tmp"
            ]
        )
        /
        df[
            "oof_edgenext_score_std_tmp"
        ]
    )


    df[
        "oof_eva_score_mean_tmp"
    ] = (
        (
            df[
                "oof_eva_score"
            ]
            -
            df[
                "oof_eva_score_mean_tmp"
            ]
        )
        /
        df[
            "oof_eva_score_std_tmp"
        ]
    )


    # --------------------------------------------
    # Frozen KMeans preprocessing from TRAIN
    # --------------------------------------------

    # Portable StandardScaler + KMeans.
    # Avoid sklearn-version-dependent pickle loading.
    scaler_data = np.load(
        PREP_DIR / "kmeans_scaler_portable.npz"
    )

    scaler_mean = scaler_data["mean"]
    scaler_scale = scaler_data["scale"]

    X_raw = df[
        top_lof_features
    ].to_numpy(dtype=np.float64)

    X_scaled = (
        X_raw - scaler_mean
    ) / scaler_scale

    cluster_centers = np.load(
        PREP_DIR / "kmeans_centers.npy"
    )

    cluster_labels = np.empty(
        len(X_scaled),
        dtype=np.int32,
    )

    chunk_size = 10000

    for start in range(
        0,
        len(X_scaled),
        chunk_size,
    ):
        end = min(
            start + chunk_size,
            len(X_scaled),
        )

        chunk = X_scaled[start:end]

        distances = (
            (
                chunk[:, None, :]
                - cluster_centers[None, :, :]
            ) ** 2
        ).sum(axis=2)

        cluster_labels[start:end] = np.argmin(
            distances,
            axis=1,
        )

    df["cluster_labels"] = cluster_labels


    # --------------------------------------------
    # Frozen TRAIN cluster mean/std
    # --------------------------------------------

    cluster_ref = (
        pd.read_parquet(
            PREP_DIR /
            "cluster_reference.parquet"
        )
    )


    df = df.merge(
        cluster_ref,
        how="left",
        on="cluster_labels",
        validate="many_to_one",
    )


    for c in cluster_cols:

        df[
            f"{c}__cluster"
        ] = (
            (
                df[c]
                -
                df[
                    f"{c}__mean"
                ]
            )
            /
            df[
                f"{c}__std"
            ]
        )


    # --------------------------------------------
    # Validate final CatBoost inputs
    # --------------------------------------------

    missing = [
        c
        for c in model_features
        if c not in df.columns
    ]

    if missing:
        raise RuntimeError(
            "Missing final model "
            f"features:\n{missing}"
        )


    # CatBoost models were reproduced
    # using categorical one-hot columns as strings.
    for c in onehot_cols:

        df[c] = (
            df[c]
            .astype(object)
            .where(
                df[c].notna(),
                "__MISSING__",
            )
            .astype(str)
        )


    print(
        "Rows:",
        len(df)
    )

    print(
        "Final CatBoost features:",
        len(model_features)
    )

    print(
        "Total NaN cells in final features:",
        int(
            df[
                model_features
            ]
            .isna()
            .sum()
            .sum()
        )
    )


    return (
        df,
        model_features,
    )


# ============================================================
# CatBoost × 45 inference
# ============================================================

def run_catboost(
    df,
    model_features,
):

    print("\n" + "=" * 90)
    print("CATBOOST × 45 INFERENCE")
    print("=" * 90)


    X = df[
        model_features
    ]


    model_paths = sorted(
        CATBOOST_DIR.glob(
            "catboost_seed*_fold*.cbm"
        )
    )


    if len(model_paths) != 45:
        raise RuntimeError(
            "Expected 45 CatBoost "
            f"models, found "
            f"{len(model_paths)}"
        )


    all_predictions = []


    for model_path in tqdm(
        model_paths
    ):

        model = (
            CatBoostClassifier()
        )

        model.load_model(
            str(model_path)
        )

        pred = (
            model.predict_proba(
                X
            )[:, 1]
        )

        all_predictions.append(
            pred
        )


    all_predictions = np.stack(
        all_predictions,
        axis=0,
    )


    final_pred = (
        all_predictions.mean(
            axis=0
        )
    )


    print(
        "Prediction range:",
        float(final_pred.min()),
        "to",
        float(final_pred.max()),
    )


    return final_pred


# ============================================================
# Main
# ============================================================

def main():

    test_meta = pd.read_csv(
        META_PATH
    )

    sample = pd.read_csv(
        SAMPLE_PATH
    )


    print(
        "Test metadata rows:",
        len(test_meta)
    )


    # --------------------------------------------
    # CNN
    # --------------------------------------------

    cnn_pred = run_cnn_inference(
        test_meta
    )


    # --------------------------------------------
    # Top-model feature table
    # --------------------------------------------
    #
    # IMPORTANT:
    # Do not write large hidden-test debug parquet files
    # during Kaggle competition reruns. Only submission.csv
    # is required for scoring.
    # --------------------------------------------

    test_features, model_features = (
        build_test_features(
            cnn_pred
        )
    )


    # --------------------------------------------
    # CatBoost
    # --------------------------------------------

    final_pred = run_catboost(
        test_features,
        model_features,
    )


    pred_df = pd.DataFrame({
        "isic_id":
            test_features[
                "isic_id"
            ].values,

        "prediction":
            final_pred,
    })


    # --------------------------------------------
    # Match sample_submission order
    # --------------------------------------------

    submission = (
        sample[
            ["isic_id"]
        ]
        .merge(
            pred_df,
            how="left",
            on="isic_id",
            validate="one_to_one",
        )
        .rename(
            columns={
                "prediction":
                    "target"
            }
        )
    )


    if submission[
        "target"
    ].isna().any():

        raise RuntimeError(
            "Submission contains "
            "missing predictions."
        )


    submission.to_csv(
        args.output,
        index=False,
    )


    print("\n" + "=" * 90)
    print("SUBMISSION COMPLETE")
    print("=" * 90)

    print("Submission rows:", len(submission))
    print(
        submission.head(3).to_string(
            index=False
        )
    )

    print("\nSaved:")
    print(
        Path(
            args.output
        ).resolve()
    )


if __name__ == "__main__":
    main()
